# Previsão da resistência à compressão do concreto

**Objetivo:** prever a resistência à compressão do concreto (em MPa) a partir da composição do traço e da idade de cura.

**Por que importa:** o ensaio de resistência leva dias (7, 28 dias ou mais). Um modelo que estima a resistência pelo traço permite ajustar a mistura antes, reduzindo retrabalho e desperdício de material na obra.

**Dados:** [Concrete Compressive Strength (Kaggle)](https://www.kaggle.com/datasets/elikplim/concrete-compressive-strength-data-set), originalmente do repositório UCI: 1.030 traços, 8 variáveis de entrada e 1 alvo.

| Variável | Unidade |
|---|---|
| cement, blast_furnace_slag, fly_ash, water, superplasticizer, coarse_aggregate, fine_aggregate | kg/m³ |
| age | dias |
| concrete_compressive_strength (**alvo**) | MPa |

## 1. Carregando os dados

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df= pd.read_csv('/kaggle/input/datasets/elikplim/concrete-compressive-strength-data-set/concrete_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/elikplim/concrete-compressive-strength-data-set/concrete_data.csv'

## 2. Primeira olhada

Início e fim da tabela, estatísticas de cada coluna e tipos de dado.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()


## 3. Qualidade dos dados

**Duplicados:** 25 linhas são cópias exatas de outras: três traços diferentes, dois deles repetidos em várias idades. Se ficassem, o modelo daria peso extra a esses traços e o teste poderia conter linhas que ele já viu no treino.

In [ ]:
df.duplicated().sum()


In [ ]:
df[df.duplicated()]


In [ ]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns))

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

**Nulos e valores impossíveis:** nenhum nulo e nenhum valor negativo.

**Checagem de domínio:** não basta o dado ser "válido", ele precisa fazer sentido físico. A idade vai de 1 a 365 dias e o cimento de 102 a 540 kg/m³, faixas plausíveis para concreto. Não há idade 0 nem consumo de cimento absurdo.

In [ ]:
df.isnull().sum()

In [ ]:
(df < 0).sum()

In [ ]:
print(df['age'].min(), df['age'].max())
print(df['cement'].min(), df['cement'].max())

## 4. Análise exploratória: o que se relaciona com a resistência?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

Ranking da correlação de cada variável com a resistência:

In [ ]:
df.corr()['concrete_compressive_strength'].sort_values(ascending=False)

**Leitura de engenharia:**

- **Cimento (+0,49)** é o fator mais ligado à resistência: mais aglomerante, concreto mais forte.
- **Superplastificante (+0,34)** e **idade (+0,34)** vêm em seguida: o aditivo permite usar menos água, e o concreto ganha resistência com o tempo de cura.
- **Água (−0,27)** é negativa: quanto mais aguado o traço, mais fraco (a relação água/cimento que todo engenheiro conhece).

Agora, os pares de variáveis correlacionados **entre si** (multicolinearidade), que é uma pergunta diferente:

In [ ]:
import itertools

colunas = df.corr().columns
pares = list(itertools.combinations(colunas, 2))  # combinations já evita repetir o par espelhado

resultado = [(c1, c2, df[c1].corr(df[c2])) for c1, c2 in pares]
corr_df = pd.DataFrame(resultado, columns=['var1', 'var2', 'correlacao'])
corr_df.reindex(corr_df['correlacao'].abs().sort_values(ascending=False).index).head(10)

O par mais forte é **água × superplastificante (−0,65)**: faz sentido, porque o superplastificante existe justamente para reduzir a água do traço. Nenhuma correlação entre variáveis é tão alta a ponto de exigir remover colunas.

## 5. Modelagem

**Separação treino/teste:** 80% dos traços para treinar e 20% guardados para avaliar, como uma prova que o modelo nunca viu.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('concrete_compressive_strength', axis=1)
y = df['concrete_compressive_strength']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Baseline: antes de qualquer modelo, uma régua mínima. O DummyRegressor ignora o traço e sempre “chuta” a mediana da resistência. É como um engenheiro que, sem olhar o traço, diz que todo concreto vai dar ~34 MPa. Se o modelo não for bem melhor que esse chute, ele não serve.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = DummyRegressor(strategy='median')
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)

mae_base = mean_absolute_error(y_test, pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, pred_baseline))
r2_base = r2_score(y_test, pred_baseline)

print(f"Baseline -> MAE: {mae_base:.2f} | RMSE: {rmse_base:.2f} | R²: {r2_base:.2f}")

**Modelos comparados:**

- **Regressão Linear** e **KNN** recebem os dados em escala (`StandardScaler`), porque usam coeficientes e distâncias, e uma variável em kg/m³ não pode pesar mais só por ter números maiores.
- **Random Forest** usa os dados originais: árvores só fazem perguntas de "maior ou menor que", então a escala não muda nada.

Métricas: **MAE** (erro médio, em MPa), **RMSE** (pune mais os erros grandes) e **R²** (quanto da variação da resistência o modelo explica).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelos = {
    "Regressão Linear": (LinearRegression(), X_train_scaled, X_test_scaled),
    "Random Forest":    (RandomForestRegressor(random_state=42), X_train, X_test),
    "KNN":              (KNeighborsRegressor(), X_train_scaled, X_test_scaled),
}

resultados = {"Baseline (Dummy)": {"MAE": mae_base, "RMSE": rmse_base, "R2": r2_base}}

for nome, (modelo, X_tr, X_te) in modelos.items():
    modelo.fit(X_tr, y_train)
    pred = modelo.predict(X_te)
    resultados[nome] = {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    }

pd.DataFrame(resultados).T

**Resultado:**

| Modelo | MAE (MPa) | RMSE (MPa) | R² |
|---|---|---|---|
| Baseline (mediana) | 13,97 | 17,42 | −0,02 |
| Regressão Linear | 8,90 | 11,19 | 0,58 |
| KNN | 7,13 | 9,19 | 0,72 |
| **Random Forest** | **3,50** | **5,26** | **0,91** |

A Regressão Linear melhora o baseline, mas fica em R² 0,58: a relação entre traço, idade e resistência não é uma reta: em parte porque o ganho de resistência com a idade, por exemplo, desacelera com o tempo. O Random Forest captura essas curvas e interações e reduz o erro médio em 4 vezes em relação ao baseline.

## Conclusão

Com o traço em mãos, dá para estimar a resistência do concreto antes do ensaio, com erro médio de ~3,5 MPa. Isso antecipa a decisão: ganha prazo de obra, evita retrabalho e, sem superdimensionar a mistura, economiza cimento, um dos insumos mais caros do traço. O modelo não substitui o ensaio, que segue obrigatório.

O Random Forest foi o melhor modelo (R² 0,91), contra ~14 MPa de erro do baseline. Cimento, idade e superplastificante são os fatores mais ligados à resistência.

**Próximos passos (v2):** ajuste de hiperparâmetros com `GridSearchCV`, validação cruzada para confirmar a estabilidade do R² e gráfico de importância das variáveis.